In [ ]:
from pathlib import Path
import json
import numpy as np
import polars as pl
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, ParameterGrid
from xgboost import XGBRegressor
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
outputPath = projectPath / 'pipelines' / 'primary' / 'output'
outputPath.mkdir(parents=True, exist_ok=True)
evaluationRows = pl.read_parquet(dataPrepPath / 'evaluationRows.parquet')
showcaseRows = pl.read_parquet(dataPrepPath / 'showcaseRows.parquet')
featureRows = pl.read_parquet(dataPrepPath / 'featureSets.parquet')
featureSets = {frame.get_column('featureSet')[0]: frame.sort('featureOrder').get_column('featureName').to_list() for frame in featureRows.partition_by('featureSet')}
rfGrid = list(ParameterGrid({'max_depth': [None, 8, 16], 'min_samples_leaf': [1, 5, 10], 'max_features': ['sqrt', 1.0]}))
xgbGrid = list(ParameterGrid({'learning_rate': [0.03, 0.1], 'max_depth': [3, 5, 7], 'min_child_weight': [1, 5]}))
makeModel = lambda modelName, parameters, seed, treeCount=None: RandomForestRegressor(n_estimators=300, random_state=seed, n_jobs=-1, **parameters) if modelName == 'rf' else XGBRegressor(n_estimators=treeCount or 2000, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=-1, early_stopping_rounds=None if treeCount else 50, **parameters)
def scoreConfiguration(modelName, parameters, trainingRows, featureNames):
    foldModel = GroupKFold(n_splits=min(3, trainingRows.get_column('window').n_unique()))
    foldScores = []
    treeCounts = []
    rowIndexes = np.arange(trainingRows.height)
    for fitIndexes, validationIndexes in foldModel.split(rowIndexes, groups=trainingRows.get_column('window').to_numpy()):
        fitRows = trainingRows[fitIndexes]
        validationRows = trainingRows[validationIndexes]
        model = makeModel(modelName, parameters, 0)
        fitX = fitRows.select(featureNames).to_numpy()
        fitY = fitRows.get_column('ndvi').to_numpy()
        validationX = validationRows.select(featureNames).to_numpy()
        validationY = validationRows.get_column('ndvi').to_numpy()
        if modelName == 'xgb':
            model.fit(fitX, fitY, eval_set=[(validationX, validationY)], verbose=False)
            treeCounts.append(model.best_iteration + 1)
        else:
            model.fit(fitX, fitY)
        foldScores.append(float(np.mean(np.abs(model.predict(validationX) - validationY))))
    return float(np.mean(foldScores)), int(np.mean(treeCounts)) if treeCounts else None
predictionRecords = []
tuningRecords = []
for outerFold in [42, 49, 55]:
    trainingRows = evaluationRows.filter((pl.col('tile') != outerFold) & (pl.col('role') == 'pool'))
    testRows = evaluationRows.filter(pl.col('tile') == outerFold)
    assert set(trainingRows.get_column('blockId').to_list()).isdisjoint(testRows.get_column('blockId').to_list())
    for featureSet, featureNames in featureSets.items():
        for modelName, searchGrid in [('rf', rfGrid), ('xgb', xgbGrid)]:
            scoredConfigurations = [(parameters, *scoreConfiguration(modelName, parameters, trainingRows, featureNames)) for parameters in searchGrid]
            chosenParameters, chosenInnerMae, chosenTreeCount = min(scoredConfigurations, key=lambda item: item[1])
            for parameters, innerMae, treeCount in scoredConfigurations:
                tuningRecords.append({'outerFold': outerFold, 'model': modelName, 'featureSet': featureSet, 'parameters': json.dumps(parameters, sort_keys=True), 'innerMae': innerMae, 'treeCount': treeCount, 'chosen': parameters == chosenParameters})
            for seed in range(5):
                model = makeModel(modelName, chosenParameters, seed, chosenTreeCount)
                model.fit(trainingRows.select(featureNames).to_numpy(), trainingRows.get_column('ndvi').to_numpy())
                targetFrames = [('mlTable', testRows)] + ([('showcaseApply', showcaseRows)] if outerFold == 42 else [])
                for sourceTable, targetRows in targetFrames:
                    predictedValues = model.predict(targetRows.select(featureNames).to_numpy())
                    predictionRecords.extend({'rowId': int(rowId), 'sourceTable': sourceTable, 'model': modelName, 'featureSet': featureSet, 'outerFold': outerFold, 'seed': seed, 'prediction': float(prediction)} for rowId, prediction in zip(targetRows.get_column('rowId'), predictedValues))
modelPredictions = pl.DataFrame(predictionRecords).sort('sourceTable', 'model', 'featureSet', 'outerFold', 'seed', 'rowId')
tuningResults = pl.DataFrame(tuningRecords).sort('outerFold', 'model', 'featureSet', 'innerMae')
scoredPredictions = modelPredictions.filter(pl.col('sourceTable') == 'mlTable').join(evaluationRows.select('rowId', 'ndvi'), on='rowId', how='left').with_columns((pl.col('prediction') - pl.col('ndvi')).alias('error'))
rfVsXgb = scoredPredictions.filter(pl.col('featureSet') == 'F0+F1').group_by('model', 'seed').agg(pl.col('error').abs().mean().alias('mae'), pl.len().alias('rowCount')).sort('model', 'seed')
modelPredictions.write_parquet(outputPath / 'modelPredictions.parquet')
tuningResults.write_parquet(outputPath / 'tuningResults.parquet')
rfVsXgb.write_parquet(outputPath / 'rfVsXgb.parquet')
assert modelPredictions.filter(pl.col('sourceTable') == 'mlTable').height == 792 * 2 * 4 * 5
assert modelPredictions.filter(pl.col('sourceTable') == 'showcaseApply').height == 52 * 2 * 4 * 5
assert tuningResults.filter(pl.col('chosen')).height == 3 * 2 * 4
assert rfVsXgb.height == 10
rfVsXgb
